# Limpieza de fuentes originales
## Señales del entorno: Violencia intrafamiliar, consumo abusivo de SPA y lesiones personales

**Equipo:** Jeisson Orlando Rodríguez, Erika Samboni y Luisa Fernanda Carbonell  
**Periodo trabajado:** 2018-2025  
**Unidad territorial:** localidad

Este notebook deja documentado solo el proceso de limpieza de las fuentes originales usadas para construir el visor. No calcula el índice VIF, no genera rankings ni hace pruebas estadísticas. Esa parte queda en el notebook analítico.

El objetivo de este archivo es responder una pregunta concreta: **qué se hizo con cada fuente antes de integrarla al tablero**.

## 1. Criterios de limpieza

Las reglas aplicadas son las mismas para las fuentes que entran al visor:

- mantener registros del periodo 2018-2025;
- normalizar nombres de localidades;
- conservar solo las 20 localidades de Bogotá para el análisis territorial;
- excluir registros sin localización de la base final territorial;
- conservar conteos originales, sin sumar fenómenos entre fuentes;
- dejar archivos limpios separados por fuente para mantener trazabilidad.

RNMC y prevalencia de SPA se documentan como insumos revisados. RNMC no entra al visor actual. Prevalencia SPA queda como insumo exploratorio.

In [15]:
from pathlib import Path
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

## 2. Rutas de trabajo

El notebook usa rutas relativas al proyecto. La ubicación recomendada para las fuentes originales es:

```text
data/raw/
```

Si el notebook se ejecuta en Google Colab y los archivos fueron cargados directamente en la raíz de la sesión, también puede encontrarlos allí. Esta opción se deja como apoyo operativo, pero la estructura recomendada para el repositorio sigue siendo `data/raw/`.


In [16]:
from pathlib import Path

def encontrar_raiz_proyecto(inicio=None):
    """
    Ubica la raíz del proyecto a partir de la carpeta actual.
    Se prioriza una carpeta que tenga data/raw o README.md.
    No usa rutas fijas del entorno.
    """
    inicio = Path(inicio or Path.cwd()).resolve()

    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw").exists():
            return carpeta
        if (carpeta / "README.md").exists():
            return carpeta

    return inicio


BASE_DIR = encontrar_raiz_proyecto()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = BASE_DIR / "outputs_limpieza"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def normalizar_nombre_archivo(nombre):
    """
    Normaliza nombres de archivo para evitar errores por mayúsculas,
    tildes o espacios repetidos.
    """
    nombre = str(nombre)
    nombre = "".join(
        c for c in unicodedata.normalize("NFKD", nombre)
        if not unicodedata.combining(c)
    )
    nombre = nombre.lower().strip()
    nombre = re.sub(r"\s+", " ", nombre)
    return nombre


def carpetas_busqueda():
    """
    Orden de búsqueda:
    1. data/raw/ dentro del repositorio.
    2. data/ dentro del repositorio.
    3. raíz del repositorio o carpeta actual.
    """
    candidatas = [
        RAW_DIR,
        DATA_DIR,
        BASE_DIR,
        Path.cwd().resolve(),
    ]

    # Evita duplicados y conserva el orden.
    vistas = []
    for carpeta in candidatas:
        carpeta = Path(carpeta).resolve()
        if carpeta.exists() and carpeta not in vistas:
            vistas.append(carpeta)

    return vistas


def buscar_archivo(nombres_posibles):
    """
    Busca un archivo por nombre exacto o por nombre normalizado.
    Permite que los archivos estén en data/raw/ o en la carpeta de trabajo.
    """
    if isinstance(nombres_posibles, str):
        nombres_posibles = [nombres_posibles]

    nombres_norm = {normalizar_nombre_archivo(nombre) for nombre in nombres_posibles}

    for carpeta in carpetas_busqueda():
        # Primero intenta coincidencia exacta.
        for nombre in nombres_posibles:
            ruta = carpeta / nombre
            if ruta.exists():
                return ruta

        # Luego intenta coincidencia no sensible a mayúsculas/tildes.
        for ruta in carpeta.iterdir():
            if ruta.is_file() and normalizar_nombre_archivo(ruta.name) in nombres_norm:
                return ruta

    buscado = ", ".join(nombres_posibles)
    revisadas = "\n".join([f"- {c}" for c in carpetas_busqueda()])
    raise FileNotFoundError(
        f"No se encontró el archivo requerido: {buscado}\n"
        f"Carpetas revisadas:\n{revisadas}\n\n"
        "Para repositorio, ubique las fuentes en data/raw/. "
        "En Colab, también puede cargar los archivos en la carpeta principal de la sesión."
    )


PATHS = {
    "vif": buscar_archivo([
        "Violencia intrafamiliar.csv",
        "violencia intrafamiliar.csv",
    ]),
    "lesiones": buscar_archivo([
        "lesiones personales.csv",
        "Lesiones personales.csv",
    ]),
    "spa_abusivo": buscar_archivo([
        "osb_saludmental_-consumoabusivo_spageneral.xlsx",
        "OSB_SALUDMENTAL_-CONSUMOABUSIVO_SPAGENERAL.xlsx",
    ]),
    "spa_prevalencia": buscar_archivo([
        "obs_prevalenciaconsumospa.xlsx",
        "OBS_PREVALENCIACONSUMOSPA.xlsx",
    ]),
    "rnmc": buscar_archivo([
        "Matriz_seguridad_RNMC_2018_2025_DataJam.xlsx",
        "matriz_seguridad_rnmc_2018_2025_datajam.xlsx",
    ]),
}

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Carpetas revisadas:")
for carpeta in carpetas_busqueda():
    print("-", carpeta)

print("\nArchivos encontrados:")
for nombre, ruta in PATHS.items():
    try:
        ruta_mostrar = ruta.relative_to(BASE_DIR)
    except ValueError:
        ruta_mostrar = ruta
    print(f"{nombre:16s} -> {ruta_mostrar}")


BASE_DIR: /content
OUTPUT_DIR: /content/outputs_limpieza
Carpetas revisadas:
- /content

Archivos encontrados:
vif              -> Violencia intrafamiliar.csv
lesiones         -> lesiones personales.csv
spa_abusivo      -> osb_saludmental_-consumoabusivo_spageneral.xlsx
spa_prevalencia  -> obs_prevalenciaconsumospa.xlsx
rnmc             -> Matriz_seguridad_RNMC_2018_2025_DataJam.xlsx


In [17]:
try:
    df_vif_test = pd.read_csv('/content/Violencia intrafamiliar.csv')
    print("File loaded successfully!")
    display(df_vif_test.head())
except FileNotFoundError:
    print("Error: The file 'Violencia intrafamiliar.csv' was not found at /content/.")
except Exception as e:
    print(f"An error occurred: {e}")

File loaded successfully!


,SEXO;ANIO;MES;HECHO;LOCALIDAD;COD_LOCALIDAD;ARMA_EMPLEADA;RANGO_VITAL;RANGO_DEL_DIA;CANTIDAD;Fecha de consulta;Datos actualizados hasta
0,FEMENINO;2018;8;VIOLENCIA INTRAFAMILIAR;SUBA;1...
1,FEMENINO;2018;3;VIOLENCIA INTRAFAMILIAR;FONTIB...
2,FEMENINO;2018;3;VIOLENCIA INTRAFAMILIAR;CIUDAD...
3,FEMENINO;2018;10;VIOLENCIA INTRAFAMILIAR;CIUDA...
4,FEMENINO;2018;10;VIOLENCIA INTRAFAMILIAR;KENNE...


## 3. Normalización de localidades

La limpieza territorial usa una llave normalizada (`loc_norm`). Se eliminan tildes, se estandarizan mayúsculas y se corrigen variantes frecuentes.

In [6]:
def quitar_tildes(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto)
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )

ALIAS_LOCALIDAD = {
    "CANDELARIA": "LA CANDELARIA",
    "LA CANDELARIA": "LA CANDELARIA",
    "MARTIRES": "LOS MARTIRES",
    "LOS MARTIRES": "LOS MARTIRES",
    "RAFAEL URIBE": "RAFAEL URIBE URIBE",
    "RAFAEL URIBE URIBE": "RAFAEL URIBE URIBE",
    "CIUDAD BOLIVAR": "CIUDAD BOLIVAR",
    "SAN CRISTOBAL": "SAN CRISTOBAL",
    "USAQUEN": "USAQUEN",
    "FONTIBON": "FONTIBON",
    "ENGATIVA": "ENGATIVA",
    "BOGOTA D.C.": "BOGOTA D.C.",
    "BOGOTA DC": "BOGOTA D.C.",
}

LOCALIDADES_VALIDAS = [
    "USAQUEN", "CHAPINERO", "SANTA FE", "SAN CRISTOBAL", "USME",
    "TUNJUELITO", "BOSA", "KENNEDY", "FONTIBON", "ENGATIVA",
    "SUBA", "BARRIOS UNIDOS", "TEUSAQUILLO", "LOS MARTIRES",
    "ANTONIO NARINO", "PUENTE ARANDA", "LA CANDELARIA",
    "RAFAEL URIBE URIBE", "CIUDAD BOLIVAR", "SUMAPAZ"
]

LOCALIDAD_DISPLAY = {
    "USAQUEN": "Usaquén",
    "CHAPINERO": "Chapinero",
    "SANTA FE": "Santa Fe",
    "SAN CRISTOBAL": "San Cristóbal",
    "USME": "Usme",
    "TUNJUELITO": "Tunjuelito",
    "BOSA": "Bosa",
    "KENNEDY": "Kennedy",
    "FONTIBON": "Fontibón",
    "ENGATIVA": "Engativá",
    "SUBA": "Suba",
    "BARRIOS UNIDOS": "Barrios Unidos",
    "TEUSAQUILLO": "Teusaquillo",
    "LOS MARTIRES": "Los Mártires",
    "ANTONIO NARINO": "Antonio Nariño",
    "PUENTE ARANDA": "Puente Aranda",
    "LA CANDELARIA": "La Candelaria",
    "RAFAEL URIBE URIBE": "Rafael Uribe Uribe",
    "CIUDAD BOLIVAR": "Ciudad Bolívar",
    "SUMAPAZ": "Sumapaz",
}

def normalizar_localidad(valor):
    if pd.isna(valor):
        return np.nan
    texto = quitar_tildes(valor).upper().strip()
    texto = re.sub(r"\s+", " ", texto)
    return ALIAS_LOCALIDAD.get(texto, texto)

def marcar_localizacion(df, col="loc_norm"):
    base = df.copy()
    base["localidad_valida"] = base[col].isin(LOCALIDADES_VALIDAS)
    base["localidad_display"] = base[col].map(LOCALIDAD_DISPLAY)
    return base

## 4. Carga de archivos originales

Se cargan los archivos sin alterar su estructura inicial. La revisión de columnas queda visible para auditoría.

In [7]:
vif_raw = pd.read_csv(PATHS["vif"], sep=";", encoding="utf-8-sig")
lesiones_raw = pd.read_csv(PATHS["lesiones"], sep=";", encoding="utf-8-sig")
spa_raw = pd.read_excel(PATHS["spa_abusivo"], sheet_name="in")
prevalencia_raw = pd.read_excel(PATHS["spa_prevalencia"], sheet_name="in")
rnmc_auditoria = pd.read_excel(PATHS["rnmc"], sheet_name="Auditoria")
rnmc_conteo = pd.read_excel(PATHS["rnmc"], sheet_name="Conteo_Escenarios")

resumen_carga = pd.DataFrame([
    {"fuente": "Violencia intrafamiliar", "filas_originales": len(vif_raw), "columnas_originales": len(vif_raw.columns)},
    {"fuente": "Lesiones personales", "filas_originales": len(lesiones_raw), "columnas_originales": len(lesiones_raw.columns)},
    {"fuente": "Consumo abusivo SPA", "filas_originales": len(spa_raw), "columnas_originales": len(spa_raw.columns)},
    {"fuente": "Prevalencia consumo SPA", "filas_originales": len(prevalencia_raw), "columnas_originales": len(prevalencia_raw.columns)},
    {"fuente": "RNMC auditoría", "filas_originales": len(rnmc_auditoria), "columnas_originales": len(rnmc_auditoria.columns)},
])

resumen_carga

,fuente,filas_originales,columnas_originales
0,Violencia intrafamiliar,54411,12
1,Lesiones personales,54656,12
2,Consumo abusivo SPA,99105,21
3,Prevalencia consumo SPA,69,7
4,RNMC auditoría,16,2


## 5. Limpieza de violencia intrafamiliar

Reglas específicas:

- se conserva el hecho `VIOLENCIA INTRAFAMILIAR`;
- se filtra 2018-2025;
- se convierte `CANTIDAD` a número;
- se normaliza localidad;
- se separan registros localizados y no localizados;
- se conserva información útil para análisis diferencial: sexo y rango vital.

In [8]:
def limpiar_base_seguridad(df, hecho_esperado, nombre_fuente):
    base = df.copy()
    base.columns = [str(c).strip() for c in base.columns]

    base["ANIO"] = pd.to_numeric(base["ANIO"], errors="coerce")
    base["MES"] = pd.to_numeric(base["MES"], errors="coerce")
    base["CANTIDAD"] = pd.to_numeric(base["CANTIDAD"], errors="coerce").fillna(0)

    base["HECHO"] = base["HECHO"].astype(str).str.upper().str.strip()
    base["SEXO"] = base["SEXO"].astype(str).str.upper().str.strip()
    base["RANGO_VITAL"] = base["RANGO_VITAL"].astype(str).str.upper().str.strip()
    base["RANGO_DEL_DIA"] = base["RANGO_DEL_DIA"].astype(str).str.upper().str.strip()
    base["ARMA_EMPLEADA"] = base["ARMA_EMPLEADA"].astype(str).str.upper().str.strip()

    base["loc_norm"] = base["LOCALIDAD"].apply(normalizar_localidad)
    base = marcar_localizacion(base)

    base = base[base["ANIO"].between(2018, 2025)].copy()
    base = base[base["HECHO"] == hecho_esperado].copy()
    base["fuente"] = nombre_fuente

    columnas = [
        "fuente", "SEXO", "ANIO", "MES", "HECHO", "LOCALIDAD", "loc_norm",
        "localidad_display", "localidad_valida", "COD_LOCALIDAD", "ARMA_EMPLEADA",
        "RANGO_VITAL", "RANGO_DEL_DIA", "CANTIDAD"
    ]
    return base[columnas]

vif_limpia = limpiar_base_seguridad(
    vif_raw,
    hecho_esperado="VIOLENCIA INTRAFAMILIAR",
    nombre_fuente="Violencia intrafamiliar"
)

vif_localizada = vif_limpia[vif_limpia["localidad_valida"]].copy()
vif_sin_localizacion = vif_limpia[~vif_limpia["localidad_valida"]].copy()

print("VIF limpia:", vif_limpia.shape)
print("VIF localizada:", vif_localizada.shape)
print("VIF sin localidad válida:", vif_sin_localizacion.shape)
print("Casos VIF localizados:", int(vif_localizada["CANTIDAD"].sum()))

vif_localizada.head()

VIF limpia: (53601, 14)
VIF localizada: (52757, 14)
VIF sin localidad válida: (844, 14)
Casos VIF localizados: 283773


,fuente,SEXO,ANIO,MES,HECHO,LOCALIDAD,loc_norm,localidad_display,localidad_valida,COD_LOCALIDAD,ARMA_EMPLEADA,RANGO_VITAL,RANGO_DEL_DIA,CANTIDAD
0,Violencia intrafamiliar,FEMENINO,2018,8,VIOLENCIA INTRAFAMILIAR,SUBA,SUBA,Suba,True,11,NO REPORTADO,ADULTEZ,MAÑANA,34
1,Violencia intrafamiliar,FEMENINO,2018,3,VIOLENCIA INTRAFAMILIAR,FONTIBÓN,FONTIBON,Fontibón,True,9,NO REPORTADO,ADULTEZ,NOCHE,9
2,Violencia intrafamiliar,FEMENINO,2018,3,VIOLENCIA INTRAFAMILIAR,CIUDAD BOLÍVAR,CIUDAD BOLIVAR,Ciudad Bolívar,True,19,NO REPORTADO,ADULTEZ,MAÑANA,16
3,Violencia intrafamiliar,FEMENINO,2018,10,VIOLENCIA INTRAFAMILIAR,CIUDAD BOLÍVAR,CIUDAD BOLIVAR,Ciudad Bolívar,True,19,NO REPORTADO,ADULTEZ,MADRUGADA,75
4,Violencia intrafamiliar,FEMENINO,2018,10,VIOLENCIA INTRAFAMILIAR,KENNEDY,KENNEDY,Kennedy,True,8,NO REPORTADO,ADULTEZ,NOCHE,61


## 6. Limpieza de lesiones personales

Se aplica la misma lógica de limpieza que en VIF para mantener consistencia entre fuentes.

In [9]:
lesiones_limpia = limpiar_base_seguridad(
    lesiones_raw,
    hecho_esperado="LESIONES PERSONALES",
    nombre_fuente="Lesiones personales"
)

lesiones_localizada = lesiones_limpia[lesiones_limpia["localidad_valida"]].copy()
lesiones_sin_localizacion = lesiones_limpia[~lesiones_limpia["localidad_valida"]].copy()

print("Lesiones limpia:", lesiones_limpia.shape)
print("Lesiones localizada:", lesiones_localizada.shape)
print("Lesiones sin localidad válida:", lesiones_sin_localizacion.shape)
print("Casos lesiones localizados:", int(lesiones_localizada["CANTIDAD"].sum()))

lesiones_localizada.head()

Lesiones limpia: (52458, 14)
Lesiones localizada: (51908, 14)
Lesiones sin localidad válida: (550, 14)
Casos lesiones localizados: 162475


,fuente,SEXO,ANIO,MES,HECHO,LOCALIDAD,loc_norm,localidad_display,localidad_valida,COD_LOCALIDAD,ARMA_EMPLEADA,RANGO_VITAL,RANGO_DEL_DIA,CANTIDAD
0,Lesiones personales,FEMENINO,2018,3,LESIONES PERSONALES,SAN CRISTÓBAL,SAN CRISTOBAL,San Cristóbal,True,4,CONTUNDENTES,ADULTEZ,TARDE,13
1,Lesiones personales,FEMENINO,2018,9,LESIONES PERSONALES,CIUDAD BOLÍVAR,CIUDAD BOLIVAR,Ciudad Bolívar,True,19,CONTUNDENTES,ADULTEZ,MADRUGADA,15
2,Lesiones personales,FEMENINO,2018,2,LESIONES PERSONALES,KENNEDY,KENNEDY,Kennedy,True,8,NO REPORTADO,ADULTEZ,TARDE,11
3,Lesiones personales,MASCULINO,2018,1,LESIONES PERSONALES,TEUSAQUILLO,TEUSAQUILLO,Teusaquillo,True,13,CONTUNDENTES,ADULTEZ,MAÑANA,9
4,Lesiones personales,FEMENINO,2018,12,LESIONES PERSONALES,LOS MÁRTIRES,LOS MARTIRES,Los Mártires,True,14,CONTUNDENTES,ADULTEZ,MADRUGADA,2


## 7. Limpieza de consumo abusivo de SPA

Reglas específicas:

- se usa `ANO` como año del registro;
- se usa `CASOS` como conteo;
- se normaliza `NOMBRELOCALIDADRESIDENCIA`;
- se filtra 2018-2025;
- se separan localidades válidas y registros sin localidad territorial útil.

In [10]:
spa_limpia = spa_raw.copy()
spa_limpia.columns = [str(c).strip() for c in spa_limpia.columns]

spa_limpia["ANO"] = pd.to_numeric(spa_limpia["ANO"], errors="coerce")
spa_limpia["MESNOTIFICACION"] = pd.to_numeric(spa_limpia["MESNOTIFICACION"], errors="coerce")
spa_limpia["CASOS"] = pd.to_numeric(spa_limpia["CASOS"], errors="coerce").fillna(0)
spa_limpia["loc_norm"] = spa_limpia["NOMBRELOCALIDADRESIDENCIA"].apply(normalizar_localidad)
spa_limpia = marcar_localizacion(spa_limpia)
spa_limpia["fuente"] = "Consumo abusivo SPA"

spa_limpia = spa_limpia[spa_limpia["ANO"].between(2018, 2025)].copy()

columnas_spa = [
    "fuente", "ANO", "MESNOTIFICACION", "SEXO", "NOMBRELOCALIDADRESIDENCIA",
    "loc_norm", "localidad_display", "localidad_valida", "CURSO_DE_VIDA",
    "TIPOASEGURAMIENTO", "NIVELEDUCATIVO", "NOMBREUPZ", "CASOS"
]
columnas_spa = [c for c in columnas_spa if c in spa_limpia.columns]
spa_limpia = spa_limpia[columnas_spa]

spa_localizada = spa_limpia[spa_limpia["localidad_valida"]].copy()
spa_sin_localizacion = spa_limpia[~spa_limpia["localidad_valida"]].copy()

print("SPA limpia:", spa_limpia.shape)
print("SPA localizada:", spa_localizada.shape)
print("SPA sin localidad válida:", spa_sin_localizacion.shape)
print("Casos SPA localizados:", int(spa_localizada["CASOS"].sum()))

spa_localizada.head()

SPA limpia: (71727, 13)
SPA localizada: (67230, 13)
SPA sin localidad válida: (4497, 13)
Casos SPA localizados: 74238


,fuente,ANO,MESNOTIFICACION,SEXO,NOMBRELOCALIDADRESIDENCIA,loc_norm,localidad_display,localidad_valida,CURSO_DE_VIDA,TIPOASEGURAMIENTO,NIVELEDUCATIVO,NOMBREUPZ,CASOS
0,Consumo abusivo SPA,2022,6,Hombre,Kennedy,KENNEDY,Kennedy,True,Juventud,Subsidiado,Secundaria completa,AMERICAS,1
1,Consumo abusivo SPA,2022,6,Hombre,Mártires,LOS MARTIRES,Los Mártires,True,Adultez,Subsidiado,Secundaria completa,LA SABANA,2
2,Consumo abusivo SPA,2022,6,Hombre,Kennedy,KENNEDY,Kennedy,True,Adolescencia,Contributivo,Secundaria incompleta,CALANDAIMA,1
3,Consumo abusivo SPA,2022,6,Hombre,Kennedy,KENNEDY,Kennedy,True,Juventud,Contributivo,Secundaria completa,LAS MARGARITAS,1
4,Consumo abusivo SPA,2022,6,Mujer,Kennedy,KENNEDY,Kennedy,True,Adultez,Contributivo,Secundaria completa,LAS MARGARITAS,1


## 8. Limpieza de prevalencia de consumo SPA

Esta fuente no entra en el índice principal. Se limpia para conservarla como insumo exploratorio de 2022.

In [11]:
prevalencia_limpia = prevalencia_raw.copy()
prevalencia_limpia.columns = [str(c).strip() for c in prevalencia_limpia.columns]

prevalencia_limpia["Año"] = pd.to_numeric(prevalencia_limpia["Año"], errors="coerce")
prevalencia_limpia["Consumo actual"] = pd.to_numeric(prevalencia_limpia["Consumo actual"], errors="coerce")
prevalencia_limpia["loc_norm"] = prevalencia_limpia["Nombre_Loc"].apply(normalizar_localidad)
prevalencia_limpia = marcar_localizacion(prevalencia_limpia)
prevalencia_limpia["fuente"] = "Prevalencia consumo SPA"

mask_spa_ilicita = prevalencia_limpia["Indicador"].astype(str).str.contains(
    "sustancia ilícita|sustancia ilicita", case=False, regex=True, na=False
)

prevalencia_spa_exploratoria = prevalencia_limpia[
    mask_spa_ilicita & prevalencia_limpia["localidad_valida"]
].copy()

print("Prevalencia SPA exploratoria:", prevalencia_spa_exploratoria.shape)
prevalencia_spa_exploratoria[["Año", "localidad_display", "Indicador", "Consumo actual"]].head()

Prevalencia SPA exploratoria: (20, 11)


,Año,localidad_display,Indicador,Consumo actual
1,2022,Usaquén,Porcentaje de personas con consumo actual de c...,9.3
2,2022,Chapinero,Porcentaje de personas con consumo actual de c...,9.7
3,2022,Santa Fe,Porcentaje de personas con consumo actual de c...,5.1
4,2022,San Cristóbal,Porcentaje de personas con consumo actual de c...,4.1
5,2022,Usme,Porcentaje de personas con consumo actual de c...,3.6


## 9. Revisión de RNMC

RNMC se revisó como fuente disponible, pero no se incorpora en el visor actual. La razón metodológica es que la base requiere tratamiento propio por artículos, numerales y temas. Se deja documentada para una fase posterior.

In [12]:
rnmc_resumen = pd.DataFrame({
    "campo": ["uso_en_version_actual", "motivo", "hojas_revisadas"],
    "valor": [
        "No entra al visor actual",
        "Requiere diccionario propio de fenómenos por artículo, numeral y tema",
        "Auditoria; Conteo_Escenarios"
    ]
})

rnmc_resumen

,campo,valor
0,uso_en_version_actual,No entra al visor actual
1,motivo,Requiere diccionario propio de fenómenos por a...
2,hojas_revisadas,Auditoria; Conteo_Escenarios


## 10. Conteos de control después de limpieza

Esta tabla permite revisar cuántos registros y casos quedan por fuente después de aplicar las reglas de limpieza.

In [13]:
resumen_limpieza = pd.DataFrame([
    {
        "fuente": "Violencia intrafamiliar",
        "registros_limpios": len(vif_limpia),
        "registros_localizados": len(vif_localizada),
        "registros_no_localizados": len(vif_sin_localizacion),
        "casos_localizados": int(vif_localizada["CANTIDAD"].sum()),
        "periodo": f"{int(vif_limpia['ANIO'].min())}-{int(vif_limpia['ANIO'].max())}",
        "entra_al_visor_actual": "Sí",
    },
    {
        "fuente": "Lesiones personales",
        "registros_limpios": len(lesiones_limpia),
        "registros_localizados": len(lesiones_localizada),
        "registros_no_localizados": len(lesiones_sin_localizacion),
        "casos_localizados": int(lesiones_localizada["CANTIDAD"].sum()),
        "periodo": f"{int(lesiones_limpia['ANIO'].min())}-{int(lesiones_limpia['ANIO'].max())}",
        "entra_al_visor_actual": "Sí",
    },
    {
        "fuente": "Consumo abusivo SPA",
        "registros_limpios": len(spa_limpia),
        "registros_localizados": len(spa_localizada),
        "registros_no_localizados": len(spa_sin_localizacion),
        "casos_localizados": int(spa_localizada["CASOS"].sum()),
        "periodo": f"{int(spa_limpia['ANO'].min())}-{int(spa_limpia['ANO'].max())}",
        "entra_al_visor_actual": "Sí",
    },
    {
        "fuente": "Prevalencia consumo SPA",
        "registros_limpios": len(prevalencia_limpia),
        "registros_localizados": len(prevalencia_spa_exploratoria),
        "registros_no_localizados": None,
        "casos_localizados": None,
        "periodo": "2022",
        "entra_al_visor_actual": "No, insumo exploratorio",
    },
    {
        "fuente": "RNMC",
        "registros_limpios": None,
        "registros_localizados": None,
        "registros_no_localizados": None,
        "casos_localizados": None,
        "periodo": "2018-2025",
        "entra_al_visor_actual": "No, fuente documentada para fase posterior",
    },
])

resumen_limpieza

,fuente,registros_limpios,registros_localizados,registros_no_localizados,casos_localizados,periodo,entra_al_visor_actual
0,Violencia intrafamiliar,53601.0,52757.0,844.0,283773.0,2018-2025,Sí
1,Lesiones personales,52458.0,51908.0,550.0,162475.0,2018-2025,Sí
2,Consumo abusivo SPA,71727.0,67230.0,4497.0,74238.0,2018-2025,Sí
3,Prevalencia consumo SPA,69.0,20.0,NaN,NaN,2022,"No, insumo exploratorio"
4,RNMC,NaN,NaN,NaN,NaN,2018-2025,"No, fuente documentada para fase posterior"


## 11. Exportación de bases limpias

Se exportan archivos separados por fuente. Esto permite revisar la limpieza sin mezclar fenómenos.

In [14]:
vif_localizada.to_csv(OUTPUT_DIR / "vif_limpia_localizada_2018_2025.csv", index=False, encoding="utf-8-sig")
lesiones_localizada.to_csv(OUTPUT_DIR / "lesiones_limpia_localizada_2018_2025.csv", index=False, encoding="utf-8-sig")
spa_localizada.to_csv(OUTPUT_DIR / "spa_abusivo_limpio_localizado_2018_2025.csv", index=False, encoding="utf-8-sig")
prevalencia_spa_exploratoria.to_csv(OUTPUT_DIR / "prevalencia_spa_exploratoria_2022.csv", index=False, encoding="utf-8-sig")
resumen_limpieza.to_csv(OUTPUT_DIR / "resumen_limpieza_fuentes.csv", index=False, encoding="utf-8-sig")
rnmc_resumen.to_csv(OUTPUT_DIR / "rnmc_revision_metodologica.csv", index=False, encoding="utf-8-sig")

print("Archivos exportados en:", OUTPUT_DIR)
for archivo in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", archivo.name)

Archivos exportados en: /content/outputs_limpieza
- lesiones_limpia_localizada_2018_2025.csv
- prevalencia_spa_exploratoria_2022.csv
- resumen_limpieza_fuentes.csv
- rnmc_revision_metodologica.csv
- spa_abusivo_limpio_localizado_2018_2025.csv
- vif_limpia_localizada_2018_2025.csv


## 12. Cierre de limpieza

Con este proceso quedan listas las fuentes localizadas para el notebook analítico. A partir de estos archivos se pueden construir agregaciones por localidad, tasas, persistencia, índices y visualizaciones.

La regla se mantiene: **las fuentes se limpian de forma separada y no se suman registros entre fenómenos**.